In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0,"/home/ws/sk6801/sw/UCSD_analysis/sandpro")
import pandas as pd
# from sklearn.linear_model import LinearRegression
# import statsmodels.api as sm



sys.path.insert(0,"../src/")
import common.d2d as d2d
from application.gain_processor_hdf5 import GainProcessor


In [ ]:
params = {
    # figure
    'figure.figsize': (15, 8),
    'figure.facecolor': 'white',  # make figure background white
    # axes
    'axes.labelsize': 20,
    'axes.linewidth': 2,
    
    'axes.titlesize': 20,
    'axes.grid.which': 'both',  # gridlines at major, minor or both ticks
    # errorbar
    'errorbar.capsize': 4,
    # font
    'font.size': 22,
    'font.family': 'Times New Roman',
    # color
    'image.cmap': 'viridis',
    # legend
    'savefig.bbox': 'tight',
    'legend.fontsize': 22,
    'legend.frameon': False,
    'legend.numpoints': 1,  # only one marker in legend
    # line
    'lines.linestyle': 'solid',
    'lines.linewidth': 2,
    'lines.markeredgewidth': 1,
    'lines.markersize': 8,
    # text
    'mathtext.default': 'regular',
    'savefig.bbox': 'tight',
    'savefig.transparent': False,
    # tick
    'xtick.top': True,  # draw ticks on the top side
    'xtick.direction': 'in',
    'xtick.labelsize': 20,
    'xtick.major.size': 8,
    'xtick.major.width': 1,
    'xtick.minor.size': 4,
    'xtick.minor.visible': False,
    'xtick.minor.width': 1,

    'ytick.right': False,  # draw ticks on the right side
    'ytick.direction': 'in',
    'ytick.labelsize': 20,
    'ytick.major.size': 6,
    'ytick.major.width': 1,
    'ytick.minor.size': 3,
    'ytick.minor.visible': False,
    'ytick.minor.width': 1,
    # GRIDS
    'grid.linestyle': '--',  ## dashed
    'mathtext.default': 'regular',
}
plt.rcParams.update(params)


In [ ]:
def create_color_scheme(color_map: str, array: object, color_range=(0,1), darken=1, reverse=False):
    values = sorted(np.unique(array))
    if reverse:
        values = values[::-1]
        
    cmap = plt.get_cmap(color_map)
    color = cmap(np.linspace(color_range[0], color_range[1], len(values)))
    
    # https://stackoverflow.com/questions/37517587/how-can-i-change-the-intensity-of-a-colormap-in-matplotlib
    color[:,0:3] *= darken
    
    clr = {values[i]: color[i] for i in range(len(values))}
    return clr

In [ ]:
gain_processor = GainProcessor()
df_GXe = gain_processor.read_run_list(run_list_path = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250608_all_run_info_single_channel.h5").get_df()
df_LXe = gain_processor.read_run_list(run_list_path = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250616_LXe_run_info_single_channel.h5").get_df()

df_all = pd.concat([df_GXe, df_LXe], axis=0)   


In [ ]:
class GainAnalysis:
    def __init__(self, df, output_path=None):
        self.df = df
        self.info = d2d.data(df)
        self.output_path = output_path
        
        self.settings()
        self.data_selection()


    def settings(self):
        self.color_temperature = create_color_scheme(
            "coolwarm_r", 
            self.info.temperature_K,
            darken = 0.8,
            color_range=(0.2,0.9),
            reverse=True)
        
        self.color_channel = create_color_scheme("viridis", 
                                    self.info.channel)

        self.color_voltage = create_color_scheme("hot", 
                                    self.info.voltage_preamp1_V,
                                    color_range=(0,0.8))

        self.dict_preamp_channel = {1: [0,1,2,3,4],
                       2: [5,6,7,8,10],
                       3: [9,11,12,13,14],
                       4: [15,16,17,18,19],
                       5: [20,21,22,23]}
        
        self.date_power_supply_changed = np.datetime64('2024-08-13')
        
    def data_selection(self):
        mask = (self.info.run_tag != "threshold_calibration")
        self.info = self.info.apply_mask(mask)



### Load Data

In [ ]:
# df_all = df_all.sort_values('date_time')

result_all = GainAnalysis(df_all, output_path = None)

result_all_df = result_all.info.get_df()

result_all_df['date_str'] = result_all_df['date_time'].apply(lambda x: str(x).split(' ')[0])
result_all_df['source'] = result_all_df['run_tag'].apply(lambda x: 
                                                         "Cs137" if "Cs" in x 
                                                         else ("Co57" if "Co" in x 
                                                               else ("LXe" if "LXe" in x
                                                                     else ("GXe" if "GXe" in x 
                                                                           else ""))))

result_all_df['unique_id'] = result_all_df['date_str'] + ' ' + result_all_df['source'].apply(str) + ' ' + result_all_df['comment'].apply(str) 
result_all_info = d2d.data(result_all_df)


### Finalized Plots

In [ ]:
class NoiseProcessor: 
    def __init__(self, result_all_info):
        self.result_all_info = result_all_info

        self.unique_id_list, idx = np.unique(self.result_all_info.unique_id.astype(str), return_index=True)

        self.unique_id_list_wrapped = []
        for unique_id in self.unique_id_list:
            if len(unique_id)>35:
                wrapped_text = unique_id.split(" ")
                position_half = int(len(wrapped_text)/2)
                #combine the two halves with a hyphen
                string = " ".join(wrapped_text[:position_half]) + "-\n" + " ".join(wrapped_text[position_half:])
                self.unique_id_list_wrapped.append(string)
            else:
                self.unique_id_list_wrapped.append(unique_id)

        self.temperature_list = self.result_all_info.temperature_K[idx]
        self.plot_ticks = np.arange(len(self.unique_id_list))
    
    def process_channel_noise(self, channel):

        self.baseline_std_list = []
        self.baseline_std_std = []
        self.baseline_mean_list = []
        self.datetime_list = []
        self.plot_ticks_list = []

        mask = (self.result_all_info.channel==channel)
        channel_info = self.result_all_info.apply_mask(mask, inplace=False)


        for j, unique_id in enumerate(self.unique_id_list):
            
            mask = channel_info.unique_id==unique_id
            tmp = channel_info.apply_mask(mask, inplace=False)

            # check if all runs have the same date
            if len(tmp.date_str) == 0:
                continue

            self.datetime_list.append(tmp.date_time[0])
            self.baseline_std_list.append(tmp.baseline_std_V.max())
            self.baseline_std_std.append(tmp.baseline_std_V.std())
            self.baseline_mean_list.append(tmp.baseline_mean_V.mean())
            self.plot_ticks_list.append(j)
        
        return

            

In [ ]:
# per channel evolution#
sep_date = np.datetime64('2024-08-27')

fig, ax = plt.subplots(1,2,figsize = (7.5, 60))

ax_top_0 = ax[0].twiny()
ax_top_1 = ax[1].twiny()

ax_top = [ax_top_0, ax_top_1]

plot_0_info = result_all_info.apply_mask(result_all_info.date_time < sep_date, inplace=False)
plot_1_info = result_all_info.apply_mask(result_all_info.date_time >= sep_date, inplace=False)

for i in range(2):
    noise_processor = NoiseProcessor(plot_0_info if i == 0 else plot_1_info)
    ax[i].plot(noise_processor.temperature_list, noise_processor.plot_ticks, 'ro-', label="Temperature", linewidth=2)

    for channel in range(24):
        noise_processor.process_channel_noise(channel)
        ax_top[i].plot(noise_processor.baseline_std_list, noise_processor.plot_ticks_list, color = result_all.color_channel[channel], label=f"Channel {channel}")


    ax[i].yaxis.set_label_position(f"{'left' if i == 0 else 'right'}")
    # ax[i].yaxis.set_label_position("right")
    ax[i].set_ylim(-1, len(noise_processor.plot_ticks)+1)
    ax[i].set_yticks(noise_processor.plot_ticks)
    ax[i].set_yticklabels(noise_processor.unique_id_list_wrapped)

    # set x axis lables
    ax[i].set_xticks(np.unique(result_all_info.temperature_K))
    ax[i].set_xticklabels([f"{i:.1f}" for i in np.unique(result_all_info.temperature_K)])
    # set ax spine and ticks color
    ax[i].spines['bottom'].set_color('red')
    ax[i].xaxis.label.set_color('red')
    ax[i].tick_params(axis='x', colors='red')
    ax[i].set_xlabel("Temperature [K]")
    ax[i].grid(which="both",axis="y")
    if i == 1:
        ax[i].invert_yaxis()

    ax_top[i].spines['bottom'].set_color('red')
    ax_top[i].set_xlabel("Baseline std [V]")
    ax_top[i].grid(which="both",axis="both")
    ax_top[i].invert_yaxis()

ax[0].yaxis.tick_left()
ax[1].yaxis.tick_right()

plt.gca().invert_yaxis()

plt.legend(loc='upper left', bbox_to_anchor=(3.5, 1), ncol=1)


In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(15,5))
plt.rcParams.update({'font.size': 18})

heatmap_array = np.zeros((3,8))

for channel in np.arange(24):

    channel_info = result_all_info.apply_mask(result_all_info.channel==channel, inplace=False)

    baseline_std_mV = channel_info.baseline_std_V.mean()*1000

    plot_row = channel % 3
    plot_col = channel // 3

    # change the rows so that it match with the physical layout of the channels
    if plot_row == 0:
        plot_row = 1
    elif plot_row == 1:
        plot_row = 0
        
    heatmap_array[plot_row, plot_col] = baseline_std_mV
        
    text = ax.text(plot_col, plot_row, f"Channel {channel}\n{baseline_std_mV:.2f}",
                    ha="center", va="center", color="black")
    
ax.xaxis.set_visible(False)
ax.yaxis.set_visible(False)
im = ax.imshow(heatmap_array, cmap='viridis',vmin=0)

plt.colorbar(im, label="Average Baseline std [mV]")
plt.tight_layout()

plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/average_noise_channel.pdf", dpi=300, bbox_inches='tight')
